In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc gem-suite matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Odhadovaná spotřeba: 3 minuty na procesoru Heron r2 (POZNÁMKA: Jedná se pouze o odhad. Skutečná doba běhu se může lišit.)*

## Pozadí

Tento tutoriál ukazuje, jak realizovat Nishimoriho fázový přechod na kvantovém procesoru IBM&reg;. Tento experiment byl původně popsán v článku [*Realizing the Nishimori transition across the error threshold for constant-depth quantum circuits*](https://arxiv.org/abs/2309.02863).

Nishimoriho fázový přechod označuje přechod mezi krátkodosahovými a dlouhodosahovými uspořádanými fázemi v náhodně svázaném Isingově modelu. Na kvantovém počítači se fáze s dlouhodobým uspořádáním projevuje jako stav, ve kterém jsou qubity provázány přes celé zařízení. Tento vysoce provázaný stav se připravuje pomocí protokolu *generování provázání měřením* (GEM). Díky využití měření uprostřed obvodu je protokol GEM schopen provázat qubity přes celé zařízení pomocí obvodů pouze konstantní hloubky. Tento tutoriál využívá implementaci protokolu GEM ze softwarového balíčku [GEM Suite](https://github.com/qiskit-community/gem-suite).

## Požadavky

Před zahájením tohoto tutoriálu se ujisti, že máš nainstalováno následující:

- Qiskit SDK v1.0 nebo novější, s podporou [vizualizace](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 nebo novější ( `pip install qiskit-ibm-runtime` )
- GEM Suite ( `pip install gem-suite` )

## Nastavení

In [2]:
import matplotlib.pyplot as plt

from collections import defaultdict

from qiskit_ibm_runtime import QiskitRuntimeService

from qiskit.transpiler import generate_preset_pass_manager

from gem_suite import PlaquetteLattice
from gem_suite.experiments import GemExperiment

## Krok 1: Přeložení klasických vstupů na kvantový problém
Protokol GEM funguje na kvantovém procesoru, jehož konektivita qubitů je popsána mřížkou. Dnešní kvantové procesory IBM používají [mřížku heavy hex](https://www.ibm.com/quantum/blog/heavy-hex-lattice). Qubity procesoru jsou seskupeny do *plaket* podle toho, které základní buňce mřížky patří. Protože qubit může patřit do více než jedné základní buňky, plakety nejsou vzájemně disjunktní. Na mřížce heavy hex má plaket 12 qubitů. Samotné plakety také tvoří mřížku, přičemž dvě plakety jsou propojeny, pokud sdílejí některé qubity. Na mřížce heavy hex sousední plakety sdílejí 3 qubity.

V softwarovém balíčku GEM Suite je základní třídou pro implementaci protokolu GEM třída `PlaquetteLattice`, která reprezentuje mřížku plaket (jež se liší od samotné mřížky heavy hex). Objekt `PlaquetteLattice` lze inicializovat z mapy párování qubitů. Aktuálně jsou podporovány pouze mapy párování heavy hex.

Následující buňka kódu inicializuje plaketovou mřížku z mapy párování kvantového procesoru IBM. Plaketová mřížka ne vždy pokrývá celý hardware. Například procesor `ibm_torino` má celkem 133 qubitů, ale největší plaketová mřížka, která se na zařízení vejde, využívá pouze 125 z nich a obsahuje celkem 18 plaket. Podobné chování lze pozorovat i u zařízení IBM Quantum&reg; s jiným počtem qubitů.

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="<YOUR_API_KEYN>", overwrite=True, set_as_default=True)
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

print(f"Number of qubits in backend: {backend.num_qubits}")
print(
    f"Number of qubits in plaquette lattice: {len(list(plaquette_lattice.qubits()))}"
)
print(f"Number of plaquettes: {len(list(plaquette_lattice.plaquettes()))}")

Number of qubits in backend: 133
Number of qubits in plaquette lattice: 125
Number of plaquettes: 18


You can visualize the plaquette lattice by generating a diagram of its graph representation. In the diagram, the plaquettes are represented by labeled hexagons, and two plaquettes are connected by an edge if they share qubits.

In [7]:
plaquette_lattice.draw_plaquettes()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif" alt="Output of the previous code cell" />

Plaketovou mřížku lze vizualizovat vygenerováním diagramu jejího grafového zobrazení. V diagramu jsou plakety reprezentovány popsanými šestiúhelníky a dvě plakety jsou propojeny hranou, pokud sdílejí qubity.

In [8]:
# Get a list of the plaquettes
plaquettes = list(plaquette_lattice.plaquettes())
# Display information about plaquette 0
plaquettes[0]

PyPlaquette(index=0, qubits=[0, 1, 2, 3, 4, 15, 16, 19, 20, 21, 22, 23], neighbors=[3, 1])

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif)

Informace o jednotlivých plaketech, například o qubitech, které obsahují, lze získat pomocí metody `plaquettes`.

In [9]:
plaquette_lattice.draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif" alt="Output of the previous code cell" />

In addition to the qubit labels and the edges indicating which qubits are connected, the diagram contains three additional pieces of information that are relevant to the GEM protocol:
- Each qubit is either shaded (gray) or unshaded. The shaded qubits are "site" qubits that represent the sites of the Ising model, and the unshaded qubits are "bond" qubits used to mediate interactions between the site qubits.
- Each site qubit is labeled either (A) or (B), indicating one of two roles a site qubit can play in the GEM protocol (the roles are explained later).
- Each edge is colored using one of six colors, thus partitioning the edges into six groups. This partitioning determines how two-qubit gates can be parallelized, as well as different scheduling patterns that are likely to incur different amounts of error on a noisy quantum processor. Because edges in a group are disjoint, a layer of two-qubit gates can be applied on those edges simultaneously. In fact, it is possible to partition the six colors into three groups of two colors such that the union of each group of two colors is still disjoint. Therefore, only three layers of two-qubit gates are needed to activate every edge. There are 12 ways to so partition the six colors, and each such partition yields a different 3-layer gate schedule.

Now that you have created a plaquette lattice, the next step is to initialize a `GemExperiment` object, passing both the plaquette lattice and the backend that you intend to run the experiment on. The `GemExperiment` class manages the actual implementation of the GEM protocol, including generating circuits, submitting jobs, and analyzing the data. The following code cell initializes the experiment class while restricting the plaquette lattice to only two of the plaquettes (21 qubits), reducing the size of the experiment to ensure that the noise in the hardware doesn't overwhelm the signal.

In [16]:
gem_exp = GemExperiment(plaquette_lattice.filter([9, 12]), backend=backend)

# visualize the plaquette lattice after filtering
plaquette_lattice.filter([9, 12]).draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif" alt="Output of the previous code cell" />

Také lze vytvořit diagram základních qubitů tvořících plaketovou mřížku.

In [12]:
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

Total number of circuits: 252


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif)

Kromě popisků qubitů a hran označujících, které qubity jsou propojeny, diagram obsahuje tři další informace relevantní pro protokol GEM:
- Každý qubit je buď stínovaný (šedý), nebo nestínovaný. Stínované qubity jsou „site" qubity reprezentující místa Isingova modelu a nestínované qubity jsou „bond" qubity sloužící k zprostředkování interakcí mezi site qubity.
- Každý site qubit je označen buď (A), nebo (B), přičemž to označuje jednu ze dvou rolí, které může site qubit v protokolu GEM hrát (role jsou vysvětleny dále).
- Každá hrana je obarvena jednou ze šesti barev, čímž jsou hrany rozděleny do šesti skupin. Toto rozdělení určuje, jak lze paralelizovat dvoiqubitové brány, a také různé plánovací vzory, které pravděpodobně způsobí různé množství chyb na zašuměném kvantovém procesoru. Protože hrany ve skupině jsou disjunktní, lze vrstvu dvoiqubitových bran aplikovat na tyto hrany současně. Ve skutečnosti je možné rozdělit šest barev do tří skupin po dvou barvách tak, aby sjednocení každé skupiny dvou barev bylo stále disjunktní. Proto jsou ke aktivaci každé hrany potřeba pouze tři vrstvy dvoiqubitových bran. Existuje 12 způsobů, jak takto rozdělit šest barev, a každé takové rozdělení poskytuje jiný 3-vrstvý plán bran.

Nyní, když jsi vytvořil plaketovou mřížku, dalším krokem je inicializace objektu `GemExperiment` předáním plaketové mřížky i backendu, na kterém chceš experiment spustit. Třída `GemExperiment` spravuje skutečnou implementaci protokolu GEM, včetně generování obvodů, odesílání úloh a analýzy dat. Následující buňka kódu inicializuje třídu experimentu a zároveň omezuje plaketovou mřížku pouze na dvě plakety (21 qubitů), čímž se zmenší rozsah experimentu, aby šum v hardwaru nezahltil signál.

In [13]:
# Restrict experiment to the first scheduling pattern
gem_exp.set_experiment_options(schedule_idx=0)

# There are less circuits now
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

# Print the RZZ angles swept over
print(f"RZZ angles:\n{gem_exp.parameters()}")

Total number of circuits: 21
RZZ angles:
[0.         0.07853982 0.15707963 0.23561945 0.31415927 0.39269908
 0.4712389  0.54977871 0.62831853 0.70685835 0.78539816 0.86393798
 0.9424778  1.02101761 1.09955743 1.17809725 1.25663706 1.33517688
 1.41371669 1.49225651 1.57079633]


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif)

Obvod protokolu GEM je sestaven pomocí následujících kroků:
1. Připrav stav všech $|+\rangle$ aplikováním Hadamardovy brány na každý qubit.
2. Aplikuj bránu $R_{ZZ}$ mezi každým párem propojených qubitů. Toho lze dosáhnout pomocí 3 vrstev bran. Každá brána $R_{ZZ}$ působí na site qubit a bond qubit. Pokud je site qubit označen (B), pak je úhel pevně nastaven na $\frac{\pi}{2}$. Pokud je site qubit označen (A), pak je úhel volný a generuje různé obvody. Ve výchozím nastavení je rozsah úhlů nastaven na 21 rovnoměrně rozmístěných bodů mezi $0$ a $\frac{\pi}{2}$ včetně.
3. Změř každý bond qubit v Pauliho bázi $X$. Protože qubity jsou měřeny v Pauliho bázi $Z$, toho lze dosáhnout aplikováním Hadamardovy brány před měřením qubitu.

Všimni si, že článek citovaný v úvodu tohoto tutoriálu používá jinou konvenci pro úhel $R_{ZZ}$, která se od konvence použité v tomto tutoriálu liší faktorem 2.

V kroku 3 jsou měřeny pouze bond qubity. Abychom pochopili, v jakém stavu zůstávají site qubity, je užitečné uvažovat případ, kdy je úhel $R_{ZZ}$ aplikovaný na site qubity (A) v kroku 2 roven $\frac{\pi}{2}$. V tomto případě jsou site qubity ponechány ve vysoce provázaném stavu podobném stavu GHZ,

$$
\lvert \text{GHZ} \rangle = \lvert 00 \cdots 00 \rangle + \lvert 11 \cdots 11 \rangle.
$$

Kvůli náhodnosti výsledků měření může být skutečný stav site qubitů jiným stavem s dlouhodobým uspořádáním, například $\lvert 00110 \rangle + \lvert 11001 \rangle$. Stav GHZ však lze obnovit aplikací dekódovací operace na základě výsledků měření. Když je úhel $R_{ZZ}$ snižován z $\frac{\pi}{2}$, dlouhodobé uspořádání lze stále obnovit až do kritického úhlu, který v nepřítomnosti šumu je přibližně $0.3 \pi$. Pod tímto úhlem výsledný stav již nevykazuje dlouhodosahové provázání. Tento přechod mezi přítomností a nepřítomností dlouhodobého uspořádání je Nishimoriho fázový přechod.

Ve výše uvedeném popisu byly site qubity ponechány neměřeny a dekódovací operaci lze provést aplikací kvantových bran. V experimentu implementovaném v GEM suite, který tento tutoriál sleduje, jsou site qubity ve skutečnosti měřeny a dekódovací operace je aplikována v klasickém post-procesním kroku.

Ve výše uvedeném popisu lze dekódovací operaci provést aplikací kvantových bran na site qubity k obnovení kvantového stavu. Pokud je však cílem okamžitě měřit stav, například pro účely charakterizace, jsou site qubity měřeny společně s bond qubity a dekódovací operaci lze aplikovat v klasickém post-procesním kroku. Takto je experiment implementován v GEM suite, které tento tutoriál sleduje.

Obvod protokolu GEM závisí kromě úhlu $R_{ZZ}$ v kroku 2, který ve výchozím nastavení přechází přes 21 hodnot, také na plánovacím vzoru použitém k implementaci 3 vrstev bran $R_{ZZ}$. Jak bylo diskutováno dříve, existuje 12 takových plánovacích vzorů. Celkový počet obvodů v experimentu je tedy $21 \times 12 = 252$.

Obvody experimentu lze generovat pomocí metody `circuits` třídy `GemExperiment`.

In [14]:
# Get the circuit at index 5
circuit = circuits[5]
# Remove the final measurements to ease visualization
circuit.remove_final_measurements()
# Draw the circuit
circuit.draw("mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif" alt="Output of the previous code cell" />

## Step 2: Optimize problem for quantum hardware execution

Transpiling quantum circuits for execution on hardware typically involves a [number of stages](/docs/guides/transpiler-stages). Typically, the stages that incur the most computational overhead are choosing the qubit layout, routing the two-qubit gates to conform to the qubit connectivity of the hardware, and optimizing the circuit to minimize its gate count and depth. In the GEM protocol, the layout and routing stages are unnecessary because the hardware connectivity is already incorporated into the design of the protocol. The circuits already have a qubit layout, and the two-qubit gates are already mapped onto native connections. Furthermore, in order to preserve the structure of the circuit as the $R_{ZZ}$ angle is varied, only very basic circuit optimization should be performed.

The `GemExperiment` class transparently transpiles circuits when executing the experiment. The layout and routing stages are already overridden by default to do nothing, and circuit optimization is performed at a level that only optimizes single-qubit gates. However, you can override or pass additional options using the `set_transpile_options` method. For the sake of visualization, the following code cell manually transpiles the circuit displayed previously, and draws the transpiled circuit.

In [15]:
# Demonstrate setting transpile options
gem_exp.set_transpile_options(
    optimization_level=1  # This is the default optimization level
)
pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(gem_exp.physical_qubits),
    **dict(gem_exp.transpile_options),
)
transpiled = pass_manager.run(circuit)
transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif" alt="Output of the previous code cell" />

Pro účely tohoto tutoriálu stačí uvažovat pouze jeden plánovací vzor. Následující buňka kódu omezuje experiment na první plánovací vzor. Výsledkem je, že experiment má pouze 21 obvodů, jeden pro každý přechodový úhel $R_{ZZ}$.

In [10]:
exp_data = gem_exp.run(shots=10_000)

To wait for the results, call the `block_for_results` method of the `ExperimentData` object. This call will cause the interpreter to hang until the jobs are finished.

In [11]:
exp_data.block_for_results()

ExperimentData(GemExperiment, d0d5880a-34c1-4aab-a7b6-c4f58516bc03, job_ids=['cwg12ptmptp00082khhg'], metadata=<5 items>, figure_names=['two_point_correlation.svg', 'normalized_variance.svg', 'plaquette_ops.svg', 'bond_ops.svg'])

Následující buňka kódu vykreslí diagram obvodu na indexu 5. Aby se zmenšila velikost diagramu, jsou měřicí brány na konci obvodu odstraněny.

In [ ]:
def magnetization_distribution(
    counts_dict: dict[str, int],
) -> dict[str, float]:
    """Compute magnetization distribution from counts dictionary."""
    # Construct dictionary from magnetization to count
    mag_dist = defaultdict(float)
    for bitstring, count in counts_dict.items():
        mag = bitstring.count("0") - bitstring.count("1")
        mag_dist[mag] += count
    # Normalize
    shots = sum(counts_dict.values())
    for mag in mag_dist:
        mag_dist[mag] /= shots
    return mag_dist


# Get counts dictionaries with and without decoding
data = exp_data.data()
# Get the last data point, which is at the angle for the GHZ state
raw_counts = data[-1]["counts"]
# Without decoding
site_indices = [
    i for i, q in enumerate(gem_exp.plaquettes.qubits()) if q.role == "Site"
]
site_raw_counts = defaultdict(int)
for key, val in raw_counts.items():
    site_str = "".join(key[-1 - i] for i in site_indices)
    site_raw_counts[site_str] += val
# With decoding
_, site_decoded_counts = gem_exp.plaquettes.decode_outcomes(
    raw_counts, return_counts=True
)

# Compute magnetization distribution
raw_magnetization = magnetization_distribution(site_raw_counts)
decoded_magnetization = magnetization_distribution(site_decoded_counts)

# Plot
plt.bar(*zip(*raw_magnetization.items()), label="raw")
plt.bar(*zip(*decoded_magnetization.items()), label="decoded", width=0.3)
plt.legend()
plt.xlabel("Magnetization")
plt.ylabel("Frequency")
plt.title("Magnetization distribution with and without decoding")

Text(0.5, 1.0, 'Magnetization distribution with and without decoding')

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/8ead3582-16df-4616-836c-bdce867ad6b8-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif)

## Krok 2: Optimalizace problému pro spuštění na kvantovém hardwaru
Transpilace kvantových obvodů pro spuštění na hardwaru obvykle zahrnuje [několik fází](/guides/transpiler-stages). Typicky jsou fáze s největší výpočetní náročností výběr rozmístění qubitů, směrování dvoiqubitových bran podle konektivity qubitů v hardwaru a optimalizace obvodu pro minimalizaci počtu bran a hloubky. V protokolu GEM jsou fáze rozmístění a směrování zbytečné, protože konektivita hardwaru je již zahrnuta v návrhu protokolu. Obvody již mají rozmístění qubitů a dvoiqubitové brány jsou již namapovány na nativní spojení. Navíc, aby byla zachována struktura obvodu při změně úhlu $R_{ZZ}$, měla by být prováděna pouze velmi základní optimalizace obvodu.

Třída `GemExperiment` transparentně transpiluje obvody při provádění experimentu. Fáze rozmístění a směrování jsou již ve výchozím nastavení přepsány tak, aby nic nedělaly, a optimalizace obvodu je prováděna na úrovni, která optimalizuje pouze jednoqubitové brány. Přesto můžeš přepsat nebo předat další možnosti pomocí metody `set_transpile_options`. Pro účely vizualizace následující buňka kódu ručně transpiluje dříve zobrazený obvod a vykreslí transpilovaný obvod.

In [13]:
exp_data.figure("two_point_correlation")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/4ecb25c8-e572-49af-a879-9943039db131-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif)

## Krok 3: Spuštění pomocí primitiv Qiskit
Pro spuštění obvodů protokolu GEM na hardwaru zavolej metodu `run` objektu `GemExperiment`. Můžeš určit počet vzorků, které chceš odebrat z každého obvodu. Metoda `run` vrací objekt [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html), který bys měl uložit do proměnné. Všimni si, že metoda `run` pouze odesílá úlohy bez čekání na jejich dokončení, jde tedy o neblokující volání.

In [14]:
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/2b351d68-3924-445a-94ef-047b16214e8a-0.avif" alt="Output of the previous code cell" />

Pro čekání na výsledky zavolej metodu `block_for_results` objektu `ExperimentData`. Toto volání způsobí, že interpret počká, dokud nejsou úlohy dokončeny.

In [15]:
gem_exp = GemExperiment(
    plaquette_lattice.filter(range(3, 9)), backend=backend
)
gem_exp.set_experiment_options(schedule_idx=0)
exp_data = gem_exp.run(shots=10_000)
exp_data.block_for_results()
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-0.avif" alt="Output of the previous code cell" />

In [16]:
gem_exp = GemExperiment(plaquette_lattice, backend=backend)
gem_exp.set_experiment_options(schedule_idx=0)
exp_data = gem_exp.run(shots=10_000)
exp_data.block_for_results()
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/37e9a4cd-6efb-4ade-ad09-8139db9d58e9-0.avif" alt="Output of the previous code cell" />

## Krok 4: Post-processing a vrácení výsledku v požadovaném klasickém formátu
Při úhlu $R_{ZZ}$ rovném $\frac{\pi}{2}$ by byl dekódovaný stav v nepřítomnosti šumu stavem GHZ. Dlouhodobé uspořádání stavu GHZ lze vizualizovat vykreslením magnetizace naměřených bitových řetězců. Magnetizace $M$ je definována jako součet jednoqubitových Pauliho operátorů $Z$,
$$
M = \sum_{j=1}^N Z_j,
$$
kde $N$ je počet site qubitů. Její hodnota pro bitový řetězec se rovná rozdílu počtu nul a počtu jedniček. Měření stavu GHZ dává stav samých nul nebo samých jedniček se stejnou pravděpodobností, takže magnetizace by byla $+N$ v polovině případů a $-N$ ve druhé polovině. V přítomnosti chyb způsobených šumem by se objevily i jiné hodnoty, ale pokud šum není příliš velký, distribuce by stále vykazovala špičky poblíž $+N$ a $-N$.

Pro surové bitové řetězce před dekódováním by distribuce magnetizace odpovídala distribuci rovnoměrně náhodných bitových řetězců v nepřítomnosti šumu.

Následující buňka kódu vykreslí magnetizaci surových bitových řetězců a dekódovaných bitových řetězců při úhlu $R_{ZZ}$ rovném $\frac{\pi}{2}$.